In [ ]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

import napari
from liffile import LifFile
from utils.io import list_images, extract_pixel_sizes_um
from cellpose import models, core, io

io.logger_setup()  # run this to get printing of progress

# Check if notebook has GPU access
if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

# Load CellposeSAM model
model = models.CellposeModel(gpu=True)

In [ ]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

#TODO: Emission ratio calculated as (donor excited donor emission)/(donor excited acceptor emission) - edCerulean_CTRL / edCitrine_FRET  

In [ ]:
# Iterate through the .czi and .nd2 files in the directory
lif_containers = list_images(RAW_DATA_DIRECTORY, format="lif")

lif_containers

In [ ]:
# Explore a different .lif file (0 defines the first file in the directory)
lif_path = lif_containers[0]

In [ ]:
# Read a single .lif container
lif_container = LifFile(lif_path)

# List all images inside the container
for img in lif_container.images:
    print(f"Image name: {img.name}, Dimensions: {img.dims}, Array Shape: {img.shape}")

In [ ]:
lif_image = lif_container.images[0].asarray()

lif_image = lif_image.transpose(1,0,2,3)
lif_image.shape

In [ ]:
# Initialize Napari Viewer
viewer = napari.Viewer(ndisplay=2)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

In [ ]:
import numpy as np# Extract pixel size (in um) data from XML metadata
x_um, y_um, z_um = extract_pixel_sizes_um(img.xml_element)

# Calculate anisotropy to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = np.mean(x_um + y_um) / z_um

In [ ]:
nuclei_labels, _ , _ = model.eval(lif_image, do_3D=True, anisotropy=rescale_factor, z_axis=0, niter=1000)
viewer.add_labels(nuclei_labels)